## Importanto os dados de treino e teste

In [1]:
# Resolve input and output paths from the supervised research root.
from pathlib import Path
import sys

_cwd = Path.cwd().resolve()
_candidates = [candidate for parent in (_cwd, *_cwd.parents)
               for candidate in (parent, parent / "machine-learning/supervised-learning")]
ROOT = next((candidate for candidate in _candidates
             if (candidate / "modelo_olimpo.py").is_file()
             and (candidate / "history/baselines/01_Data_Prep.ipynb").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from within the Olimpo repository.")
DATA = ROOT / "data"
HISTORY_RESULTS = ROOT / "history/resultados"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Original experiment starts here.
import joblib

In [2]:
X_tr, X_te, y_tr, y_te = joblib.load((DATA / 'dados_preparados.pkl'))

## Importando os transformers

In [3]:
configs = joblib.load((DATA / 'transformers.pkl'))
print(type(configs))

<class 'dict'>


## Imports

In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_validate

# Treinamento dos modelos

## SVM - TF-IDF de palavras

Aqui comparamos dois métodos de **validação**, usando o mesmo modelo nos dois: `LinearSVC` com a configuração `word` (TF-IDF de palavras, unigramas e bigramas).

### Para que serve a validação

Queremos saber como o modelo vai se sair em notícias que ele nunca viu. Medir no próprio treino não serve: o modelo já viu aqueles exemplos e tende a acertar quase tudo (aqui o F1 no treino chega a 1.0). Então separamos uma parte dos dados que o modelo **não usa para treinar** e medimos nela.

Nesse projeto, os dados são divididos em três papéis:
- **Treino** (`X_tr`): usado para treinar e para validar, seja por K-Fold ou Hold-out
- **Validação**: um pedaço do treino separado temporariamente para estimar o desempenho e comparar configurações/modelos
- **Teste** (`X_te`): separado lá no `01_Data_Prep` e usado só no final, para a avaliação definitiva. Ele não entra nas comparações abaixo; se entrasse, a escolha do modelo passaria a ser feita "olhando" para o teste e ele deixaria de ser uma medida honesta

Os dois métodos abaixo são formas diferentes de criar essa validação a partir do treino.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.svm import LinearSVC
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, f1_score,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split

scoring = ["f1_macro", "accuracy", "precision_macro", "recall_macro", "roc_auc"]

def criar_pipe():
    return Pipeline([
        ("prep", ColumnTransformer(configs["word"])),
        ("svm", LinearSVC(class_weight="balanced", random_state=42, max_iter=2000)),
    ])

### K-Fold (validação cruzada)

**Como funciona:**
1. O conjunto de treino é dividido em *k* partes (folds) de tamanho parecido. Aqui *k* = 5
2. O modelo é treinado *k* vezes. Em cada rodada, um fold diferente fica de fora como validação e os outros *k*-1 são usados para treinar
3. Ao final, todo exemplo foi usado como validação exatamente uma vez, e temos *k* medidas de cada métrica. O resultado é a **média** delas, com o **desvio padrão**

```
Rodada 1: [VAL] [tr ] [tr ] [tr ] [tr ]
Rodada 2: [tr ] [VAL] [tr ] [tr ] [tr ]
Rodada 3: [tr ] [tr ] [VAL] [tr ] [tr ]
Rodada 4: [tr ] [tr ] [tr ] [VAL] [tr ]
Rodada 5: [tr ] [tr ] [tr ] [tr ] [VAL]
```

**Detalhes usados aqui:**
- `StratifiedKFold`: cada fold mantém a mesma proporção de fake/real do conjunto completo
- `shuffle=True` com `random_state=42`: embaralha antes de dividir, de forma reproduzível
- `return_train_score=True`: também mede no treino de cada rodada, para comparar com a validação e detectar overfitting
- A matriz de confusão usa `cross_val_predict`, que junta as predições de cada exemplo feitas pelo modelo em que ele estava na validação

**Para que serve:** dar uma estimativa mais **estável e confiável** do desempenho, porque não depende de uma única divisão dos dados. O desvio padrão mostra o quanto o resultado varia entre divisões. É o método preferido para comparar modelos/configurações quando o dataset não é muito grande.

**Custo:** treina o modelo *k* vezes (aqui, 5x mais lento que o Hold-out).

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_res = cross_validate(criar_pipe(), X_tr, y_tr, cv=cv, scoring=scoring, return_train_score=True)

print("SVM (word) - K-Fold")
kfold = {"f1_macro_treino": cv_res["train_f1_macro"].mean()}
for m in scoring:
    vals = cv_res[f"test_{m}"]
    print(f"  {m}: {vals.mean():.4f} ± {vals.std():.4f}")
    kfold[m] = vals.mean()
print(f"  f1_macro_treino: {kfold['f1_macro_treino']:.4f}")

# Matriz de confusão com as predições out-of-fold
y_pred_cv = cross_val_predict(criar_pipe(), X_tr, y_tr, cv=cv)
fig, ax = plt.subplots(figsize=(5, 5))
ConfusionMatrixDisplay.from_predictions(y_tr, y_pred_cv, cmap="Blues", ax=ax)
ax.set_title("Matriz de Confusão (K-Fold): word")
plt.show()

#### Por que 5 folds?

A escolha de *k* é um equilíbrio entre três coisas:
- **Viés (pessimismo):** cada rodada treina com (*k*-1)/*k* dos dados. Com *k* pequeno o modelo treina com menos dados do que terá no final, e a estimativa fica pessimista. Com *k* = 2, por exemplo, **cada modelo** treina com só metade do treino (no conjunto das rodadas todos os dados são usados, mas nenhum modelo individual vê mais que a metade)
- **Variância da estimativa:** com *k* grande, cada fold de validação fica pequeno e a métrica de cada rodada oscila mais. Além disso, os modelos das rodadas treinam com dados quase idênticos, então a média não fica tão mais estável quanto se esperaria
- **Custo:** o modelo é treinado *k* vezes

Os valores mais usados na literatura são *k* = 5 e *k* = 10 (Kohavi, 1995; Hastie, Tibshirani e Friedman, *The Elements of Statistical Learning*, seção 7.10). Para confirmar a escolha **neste dataset**, o teste abaixo roda o K-Fold com *k* = 2, 3, 5, 10 e 20. Cada *k* é repetido 5 vezes com embaralhamentos diferentes (`random_state` de 0 a 4), e para cada um medimos:
- `f1_medio`: F1 macro médio (para ver o viés)
- `std_entre_folds`: quanto o F1 varia de um fold para outro dentro da mesma validação cruzada
- `std_entre_repeticoes`: quanto a **média final** muda só por trocar o embaralhamento, ou seja, o quanto o resultado é confiável
- `tempo_por_cv_s`: tempo de uma validação cruzada completa

⚠️ Esta célula demora alguns minutos (treina o modelo 200 vezes).

In [ ]:
import time
import numpy as np
from sklearn.model_selection import cross_val_score

linhas = []
for k in [2, 3, 5, 10, 20]:
    medias, stds, inicio = [], [], time.time()
    for rep in range(5):
        s = cross_val_score(
            criar_pipe(), X_tr, y_tr,
            cv=StratifiedKFold(n_splits=k, shuffle=True, random_state=rep),
            scoring="f1_macro", n_jobs=-1,
        )
        medias.append(s.mean())
        stds.append(s.std())
    linhas.append({
        "k": k,
        "treino_por_fold": int(len(X_tr) * (k - 1) / k),
        "val_por_fold": len(X_tr) // k,
        "f1_medio": np.mean(medias),
        "std_entre_folds": np.mean(stds),
        "std_entre_repeticoes": np.std(medias),
        "tempo_por_cv_s": (time.time() - inicio) / 5,
    })

teste_k = pd.DataFrame(linhas)
teste_k.round(4)

**Resultado do teste** (tempo medido numa máquina com 2 núcleos):

| k | treino por fold | validação por fold | F1 médio | std entre folds | std entre repetições | tempo (s) |
|---|---|---|---|---|---|---|
| 2 | 2880 | 2880 | 0.9033 | 0.0033 | 0.0034 | 2.9 |
| 3 | 3840 | 1920 | 0.9109 | 0.0050 | 0.0036 | 2.6 |
| **5** | **4608** | **1152** | **0.9139** | **0.0051** | **0.0022** | **4.1** |
| 10 | 5184 | 576 | 0.9152 | 0.0087 | 0.0021 | 8.0 |
| 20 | 5472 | 288 | 0.9165 | 0.0156 | 0.0007 | 16.8 |

**Leitura:**
- **Viés:** de *k* = 2 para *k* = 5 o F1 sobe 0.011. Com 2 ou 3 folds o modelo treina com poucos dados e a estimativa fica claramente pessimista. De 5 para 10 o ganho é só 0.0013, menor que a variação entre repetições (≈0.002), ou seja, não dá para distinguir de ruído. A partir de 5 o viés já é pequeno
- **Estabilidade:** *k* = 5 e *k* = 10 têm praticamente a mesma variação entre repetições (0.0022 x 0.0021). Dobrar o custo indo para 10 não deixa o resultado mais confiável
- **Custo:** *k* = 10 leva o dobro do tempo e *k* = 20 leva 4x, por um ganho que cabe dentro do ruído
- **Tamanho da validação:** com 5 folds, cada validação tem 1152 notícias, o bastante para cada métrica e matriz de confusão por fold serem representativas. Com 20 folds são só 288, e o F1 de um fold para outro varia 3x mais
- **Comparação com o Hold-out:** o Hold-out deste notebook separa 80/20, exatamente o mesmo tamanho de treino e validação de cada rodada do K-Fold com 5 folds. Assim, os dois métodos treinam o modelo com a mesma quantidade de dados, e a comparação entre eles fica justa

**Conclusão:** *k* = 5 é o menor *k* que já tira quase todo o pessimismo, tem a mesma estabilidade de *k* = 10 pela metade do custo, e deixa o K-Fold diretamente comparável ao Hold-out 80/20.

Obs.: a mesma tabela mostra que só o embaralhamento já mexe o F1 médio em torno de ±0.002. Diferenças entre modelos ou configurações menores que isso não devem ser tratadas como melhora real.

### Hold-out

**Como funciona:**
1. O conjunto de treino é dividido **uma única vez** em duas partes: treino (80%) e validação (20%), de forma estratificada
2. O modelo é treinado uma vez na parte de treino
3. As métricas são calculadas uma vez na parte de validação

```
[          treino (80%)          ] [ validação (20%) ]
```

**Para que serve:** é a forma mais simples e rápida de validar. Faz sentido quando:
- o dataset é grande o bastante para que 20% já seja uma amostra representativa
- treinar o modelo é caro (redes neurais, datasets enormes) e repetir *k* vezes não é viável
- queremos só uma checagem rápida durante o desenvolvimento

**Limitações:**
- O resultado depende de **qual** divisão foi sorteada. Outra `random_state` pode dar um número um pouco diferente, e não temos como saber a variação com uma medida só
- 20% do treino nunca é usado para treinar, e 80% nunca é usado para validar

In [ ]:
X_fit, X_val, y_fit, y_val = train_test_split(
    X_tr, y_tr, test_size=0.2, stratify=y_tr, random_state=42
)
print(f"Treino: {X_fit.shape[0]} | Validação: {X_val.shape[0]}")

pipe_holdout = criar_pipe().fit(X_fit, y_fit)
y_pred_val = pipe_holdout.predict(X_val)

holdout = {
    "f1_macro_treino": f1_score(y_fit, pipe_holdout.predict(X_fit), average="macro"),
    "f1_macro": f1_score(y_val, y_pred_val, average="macro"),
    "accuracy": accuracy_score(y_val, y_pred_val),
    "precision_macro": precision_score(y_val, y_pred_val, average="macro"),
    "recall_macro": recall_score(y_val, y_pred_val, average="macro"),
    # LinearSVC não tem predict_proba; para AUC usa decision_function
    "roc_auc": roc_auc_score(y_val, pipe_holdout.decision_function(X_val)),
}

print("\nSVM (word) - Hold-out")
for m, v in holdout.items():
    print(f"  {m}: {v:.4f}")

fig, ax = plt.subplots(figsize=(5, 5))
ConfusionMatrixDisplay.from_predictions(y_val, y_pred_val, cmap="Blues", ax=ax)
ax.set_title("Matriz de Confusão (Hold-out): word")
plt.show()

### Comparação K-Fold x Hold-out

| | K-Fold | Hold-out |
|---|---|---|
| Divisões | *k* (aqui 5) | 1 |
| Treinamentos | *k* | 1 |
| Resultado | média ± desvio padrão | um valor único |
| Uso dos dados | todo exemplo treina e valida | 80% só treina, 20% só valida |
| Estabilidade da estimativa | alta | depende da divisão sorteada |
| Custo computacional | maior | menor |
| Quando usar | datasets pequenos/médios, comparar modelos | datasets grandes, modelos caros, checagem rápida |

**Como ler a tabela abaixo:** a coluna `diferença` é Hold-out menos a média do K-Fold. Se ela for menor que o desvio padrão do K-Fold (`K-Fold (std)`), o Hold-out caiu dentro da variação normal entre divisões, e os dois métodos concordam. Vale também comparar `f1_macro_treino` com `f1_macro`: uma diferença grande entre eles indica overfitting.

In [ ]:
comparacao = pd.DataFrame({"K-Fold (média)": kfold, "Hold-out": holdout})
comparacao["K-Fold (std)"] = pd.Series({m: cv_res[f"test_{m}"].std() for m in scoring})
comparacao["diferença"] = comparacao["Hold-out"] - comparacao["K-Fold (média)"]
comparacao.round(4)